#### Importar librerias

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#### Descargar los datos

In [2]:
tickers = ["SPY", "EEM", "AGG", "GLD"]

datos = yf.download(
    tickers,
    start="2015-01-01",
    end="2025-01-01",
    auto_adjust=True
)

precios = datos["Close"]
precios.head()

[*********************100%***********************]  4 of 4 completed


Ticker,AGG,EEM,GLD,SPY
Date,,,,
2015-01-02,80.367279,30.198092,114.080002,169.687836
2015-01-05,80.541946,29.660643,115.800003,166.623337
2015-01-06,80.745720,29.536020,117.120003,165.053940
2015-01-07,80.731140,30.174717,116.430000,167.110687
2015-01-08,80.607430,30.688801,115.940002,170.076065


#### Comprobar nulos

In [3]:
precios.isna().sum()

Ticker
AGG    0
EEM    0
GLD    0
SPY    0
dtype: int64

#### Rentabilidades diarias

In [4]:
ret_simple = precios.pct_change().dropna()
ret_log = np.log(precios / precios.shift(1)).dropna()

ret_simple.head()

Ticker,AGG,EEM,GLD,SPY
Date,,,,
2015-01-05,0.002173,-0.017797,0.015077,-0.018060
2015-01-06,0.002530,-0.004202,0.011399,-0.009419
2015-01-07,-0.000181,0.021624,-0.005891,0.012461
2015-01-08,-0.001532,0.017037,-0.004209,0.017745
2015-01-09,0.002437,-0.003300,0.011385,-0.008014


#### Comprobar que simple es similar a logarítmica

In [5]:
comparacion = pd.DataFrame({
    "simple": ret_simple["SPY"],
    "log": ret_log["SPY"],
    "diferencia": (ret_simple["SPY"] - ret_log["SPY"]).abs()
})
comparacion.head(10)


,simple,log,diferencia
Date,,,
2015-01-05,-0.018060,-0.018225,0.000165
2015-01-06,-0.009419,-0.009463,0.000045
2015-01-07,0.012461,0.012384,0.000077
2015-01-08,0.017745,0.017589,0.000156
2015-01-09,-0.008014,-0.008046,0.000032
2015-01-12,-0.007834,-0.007864,0.000031
2015-01-13,-0.002813,-0.002816,0.000004
2015-01-14,-0.006037,-0.006056,0.000018
2015-01-15,-0.009161,-0.009203,0.000042


In [6]:
print(comparacion["diferencia"].max())

0.006462801599780882


#### Estadísticos anualizados

In [7]:
media_anual = ret_simple.mean() * 252
vol_anual = ret_simple.std() * np.sqrt(252)

resumen = pd.DataFrame({
    "Rentabilidad anual": media_anual,
    "Volatilidad anual": vol_anual
})
resumen

,Rentabilidad anual,Volatilidad anual
Ticker,,
AGG,0.014250,0.053105
EEM,0.051199,0.206767
GLD,0.085389,0.141207
SPY,0.138110,0.176192


#### Matriz de correlaciones

In [8]:
correlaciones = ret_simple.corr()
correlaciones

Ticker,AGG,EEM,GLD,SPY
Ticker,,,,
AGG,1.000000,0.108548,0.373408,0.093993
EEM,0.108548,1.000000,0.157576,0.755217
GLD,0.373408,0.157576,1.000000,0.046814
SPY,0.093993,0.755217,0.046814,1.000000


 #### Diversificación

In [9]:
# Los dos activos menos correlacionados
a, b = "SPY", "GLD"

# Volatilidad anual de cada uno por separado
vol_a = ret_simple[a].std() * np.sqrt(252)
vol_b = ret_simple[b].std() * np.sqrt(252)

# Correlación entre ellos
rho = ret_simple[[a, b]].corr().iloc[0, 1]

# Cartera 50/50
w_a, w_b = 0.5, 0.5

# Fórmula de volatilidad de una cartera de 2 activos
vol_cartera = np.sqrt(
    (w_a**2) * (vol_a**2) +
    (w_b**2) * (vol_b**2) +
    2 * w_a * w_b * rho * vol_a * vol_b
)

print(f"Volatilidad {a} sola:       {vol_a:.4f}  ({vol_a*100:.2f}%)")
print(f"Volatilidad {b} sola:       {vol_b:.4f}  ({vol_b*100:.2f}%)")
print(f"Media simple de las dos:   {(vol_a+vol_b)/2:.4f}  ({(vol_a+vol_b)/2*100:.2f}%)")
print(f"Volatilidad cartera 50/50: {vol_cartera:.4f}  ({vol_cartera*100:.2f}%)")

Volatilidad SPY sola:       0.1762  (17.62%)
Volatilidad GLD sola:       0.1412  (14.12%)
Media simple de las dos:   0.1587  (15.87%)
Volatilidad cartera 50/50: 0.1154  (11.54%)


#### Preparar datos para el optimizador

In [10]:
from pypfopt import expected_returns, risk_models

# Rentabilidad esperada anualizada de cada activo
mu = expected_returns.mean_historical_return(precios)

# Matriz de covarianzas anualizada
S = risk_models.sample_cov(precios)

print("Rentabilidades esperadas (μ):")
print(mu)
print("\nMatriz de covarianzas (S):")
print(S)

Rentabilidades esperadas (μ):
Ticker
AGG    0.012919
EEM    0.030161
GLD    0.078323
SPY    0.130314
dtype: float64

Matriz de covarianzas (S):
Ticker       AGG       EEM       GLD       SPY
Ticker                                        
AGG     0.002820  0.001192  0.002800  0.000879
EEM     0.001192  0.042753  0.004601  0.027513
GLD     0.002800  0.004601  0.019940  0.001165
SPY     0.000879  0.027513  0.001165  0.031044


#### Frontera eficiente y cartera de máximo Sharpe

In [11]:
from pypfopt import EfficientFrontier

# Creamos el objeto de optimización con nuestros ingredientes
ef = EfficientFrontier(mu, S)

# Le pedimos la cartera que MAXIMIZA el ratio de Sharpe
pesos = ef.max_sharpe(risk_free_rate=0.02)

# Limpiamos los pesos (redondea los casi-cero a cero)
pesos_limpios = ef.clean_weights()

print("Cartera de máximo Sharpe:")
for activo, peso in pesos_limpios.items():
    print(f"  {activo}: {peso*100:.2f}%")

Cartera de máximo Sharpe:
  AGG: 0.00%
  EEM: 0.00%
  GLD: 44.10%
  SPY: 55.89%


In [12]:
rendimiento = ef.portfolio_performance(verbose=True, risk_free_rate=0.02)

Expected annual return: 10.7%
Annual volatility: 11.9%
Sharpe Ratio: 0.73


#### Cartera con mínimos y máximos por activo

In [13]:
from pypfopt import EfficientFrontier

# Nuevo optimizador (hay que recrearlo: el anterior ya se "gastó" al resolver)
ef_r = EfficientFrontier(mu, S)

# Restricción 1: cada activo entre 5% y 40%
#   -> obliga a que TODOS entren (mínimo 5%)
#   -> impide concentración excesiva (máximo 40%)
ef_r.add_constraint(lambda w: w >= 0.05)
ef_r.add_constraint(lambda w: w <= 0.40)

# Volvemos a maximizar Sharpe, ahora con las restricciones puestas
pesos_r = ef_r.max_sharpe(risk_free_rate=0.02)
pesos_r_limpios = ef_r.clean_weights()

print("Cartera de máximo Sharpe CON restricciones:")
for activo, peso in pesos_r_limpios.items():
    print(f"  {activo}: {peso*100:.2f}%")

print("\nRendimiento:")
ef_r.portfolio_performance(verbose=True, risk_free_rate=0.02)

Cartera de máximo Sharpe CON restricciones:
  AGG: 15.00%
  EEM: 5.00%
  GLD: 40.00%
  SPY: 40.00%

Rendimiento:
Expected annual return: 8.7%
Annual volatility: 10.2%
Sharpe Ratio: 0.65


(np.float64(0.08690056380538048),
 np.float64(0.10219745967550369),
 np.float64(0.6546206140328973))

#### El motor de optimización como función

In [14]:
def optimizar_cartera(mu, S, restricciones=None, risk_free_rate=0.02):
    """
    Calcula la cartera de máximo Sharpe a partir de rentabilidades
    esperadas (mu) y matriz de covarianzas (S).

    Parámetros
    ----------
    mu : pd.Series
        Rentabilidades esperadas anualizadas por activo.
    S : pd.DataFrame
        Matriz de covarianzas anualizada.
    restricciones : list[callable], opcional
        Lista de funciones lambda con restricciones sobre los pesos.
        Ej: [lambda w: w >= 0.05, lambda w: w <= 0.40]
    risk_free_rate : float
        Tasa libre de riesgo para el cálculo del Sharpe.

    Devuelve
    -------
    dict
        Pesos limpios por activo y métricas de rendimiento.
    """
    ef = EfficientFrontier(mu, S)

    # Aplicar restricciones si las hay
    if restricciones:
        for r in restricciones:
            ef.add_constraint(r)

    # Optimizar
    ef.max_sharpe(risk_free_rate=risk_free_rate)
    pesos = ef.clean_weights()

    # Métricas
    rent, vol, sharpe = ef.portfolio_performance(risk_free_rate=risk_free_rate)

    return {
        "pesos": pesos,
        "rentabilidad": rent,
        "volatilidad": vol,
        "sharpe": sharpe,
    }

#### Probar que la función funciona igual que antes

In [15]:
# Sin restricciones -> debe dar el resultado de la celda 9/10
r1 = optimizar_cartera(mu, S)
print("SIN restricciones:", r1["pesos"])
print(f"  Sharpe: {r1['sharpe']:.2f}\n")

# Con restricciones -> debe dar el resultado de la celda 11
restr = [lambda w: w >= 0.05, lambda w: w <= 0.40]
r2 = optimizar_cartera(mu, S, restricciones=restr)
print("CON restricciones:", r2["pesos"])
print(f"  Sharpe: {r2['sharpe']:.2f}")

SIN restricciones: OrderedDict([('AGG', 0.0), ('EEM', 0.0), ('GLD', 0.44105), ('SPY', 0.55895)])
  Sharpe: 0.73

CON restricciones: OrderedDict([('AGG', 0.15), ('EEM', 0.05), ('GLD', 0.4), ('SPY', 0.4)])
  Sharpe: 0.65


#### Importar Black-Litterman y definir pesos de mercadp

In [16]:
from pypfopt import black_litterman
from pypfopt.black_litterman import BlackLittermanModel

# Pesos de mercado aproximados (proxy por tamaño relativo de cada clase de activo)
# En un caso real vendrían de la capitalización/AUM de cada ETF.
market_caps = {
    "AGG": 100e9,   # bonos: mercado enorme
    "EEM": 20e9,    # emergentes
    "GLD": 60e9,    # oro
    "SPY": 400e9,   # bolsa EEUU: el más grande
}

print("Pesos de mercado implícitos:")
total = sum(market_caps.values())
for activo, cap in market_caps.items():
    print(f"  {activo}: {cap/total*100:.1f}%")

Pesos de mercado implícitos:
  AGG: 17.2%
  EEM: 3.4%
  GLD: 10.3%
  SPY: 69.0%


#### Reverse optimization

In [20]:
# delta: aversión al riesgo del mercado (valor estándar ~2.5)
delta = 2.5

# La matriz de covarianzas S ya la tenemos de antes
# reverse optimization: deduce las rentabilidades implícitas del mercado
rent_equilibrio = black_litterman.market_implied_prior_returns(
    market_caps, delta, S
)

print("Rentabilidades de equilibrio (prior):")
print(rent_equilibrio)

Rentabilidades de equilibrio (prior):
Ticker
AGG    0.003559
EEM    0.052826
GLD    0.008768
SPY    0.056576
dtype: float64


#### Comprobar que el prior da una cartera sensata

In [22]:
from pypfopt.efficient_frontier import EfficientFrontier

ef_check = EfficientFrontier(rent_equilibrio, S)
ef_check.max_quadratic_utility(risk_aversion=delta)  # mismo criterio que el prior
pesos_check = ef_check.clean_weights()

print("Cartera con el prior de equilibrio (criterio coherente):")
for activo, peso in pesos_check.items():
    print(f"  {activo}: {peso*100:.2f}%")

print("\nPesos de mercado originales (para comparar):")
total = sum(market_caps.values())
for activo, cap in market_caps.items():
    print(f"  {activo}: {cap/total*100:.1f}%")

Cartera con el prior de equilibrio (criterio coherente):
  AGG: 17.24%
  EEM: 3.45%
  GLD: 10.35%
  SPY: 68.97%

Pesos de mercado originales (para comparar):
  AGG: 17.2%
  EEM: 3.4%
  GLD: 10.3%
  SPY: 69.0%


#### Añadir una view y calcular el posterior

In [23]:
# Definimos nuestra view: EEM rentará un 12% anual
viewdict = {
    "EEM": 0.12
}

# Construimos el modelo Black-Litterman
bl = BlackLittermanModel(
    S,                                  # matriz de covarianzas
    pi=rent_equilibrio,                 # el prior (equilibrio de mercado)
    absolute_views=viewdict,            # nuestras views absolutas
    omega="default"                     # confianza automática (por defecto)
)

# El posterior: rentabilidades ajustadas (prior + view)
rent_posterior = bl.bl_returns()

# Comparamos prior vs posterior
comparacion_bl = pd.DataFrame({
    "Prior (equilibrio)": rent_equilibrio,
    "Posterior (con view)": rent_posterior,
    "Cambio": rent_posterior - rent_equilibrio
})
print(comparacion_bl)

        Prior (equilibrio)  Posterior (con view)    Cambio
Ticker                                                    
AGG               0.003559              0.004495  0.000936
EEM               0.052826              0.086413  0.033587
GLD               0.008768              0.012383  0.003614
SPY               0.056576              0.078190  0.021615


#### Optimizar con el posterior de Black-Litterman

In [24]:
# Metemos el posterior en NUESTRO motor (con restricciones sensatas)
restr = [lambda w: w >= 0.02, lambda w: w <= 0.50]
resultado_bl = optimizar_cartera(rent_posterior, S, restricciones=restr)

print("Cartera Black-Litterman (con view alcista en EEM):")
for activo, peso in resultado_bl["pesos"].items():
    print(f"  {activo}: {peso*100:.2f}%")
print(f"\n  Rentabilidad: {resultado_bl['rentabilidad']*100:.1f}%")
print(f"  Volatilidad:  {resultado_bl['volatilidad']*100:.1f}%")
print(f"  Sharpe:       {resultado_bl['sharpe']:.2f}")

Cartera Black-Litterman (con view alcista en EEM):
  AGG: 2.00%
  EEM: 46.00%
  GLD: 2.00%
  SPY: 50.00%

  Rentabilidad: 7.9%
  Volatilidad:  17.2%
  Sharpe:       0.34
